<a href="https://colab.research.google.com/github/jelly-drink/IGEL-CPs/blob/main/How%20Are%20You%20Today.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Welcome to How Are You Today! 🌷")
print("Bagaimana perasaanmu hari ini?")

Welcome to How Are You Today! 🌷
Bagaimana perasaanmu hari ini?


In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML
from datetime import datetime
import numpy as np
import io
import wave
import base64

# =====================================
# 1. TEMA DAN WARNA APLIKASI
# =====================================

themes = {
    "Soft Sage (Default)": {
        "bg_main": "#FAF8F1",
        "border": "#DCE8D7",
        "text_main": "#526B55",
        "button_bg": "#DCE8D7",
        "button_text": "#344B36",
        "button_hover": "#C3D8BB",
        "bubble_bg": "#EAF0E5"
    },
    "Sweet Sakura": {
        "bg_main": "#FFF5F5",
        "border": "#FAD0C4",
        "text_main": "#9B4F54",
        "button_bg": "#FAD0C4",
        "button_text": "#6B3033",
        "button_hover": "#F8B5A5",
        "bubble_bg": "#FFE3E3"
    },
    "Calm Ocean": {
        "bg_main": "#F0F8FF",
        "border": "#BDE0FE",
        "text_main": "#2A5C8A",
        "button_bg": "#BDE0FE",
        "button_text": "#1D3E5E",
        "button_hover": "#A2D2FF",
        "bubble_bg": "#EAF2F8"
    },
    "Cozy Lavender": {
        "bg_main": "#FAF5FF",
        "border": "#E8DBFD",
        "text_main": "#5E3A8C",
        "button_bg": "#E8DBFD",
        "button_text": "#412463",
        "button_hover": "#D6C1FC",
        "bubble_bg": "#F3EBFF"
    }
}

moods = {
    "😊 Bahagia": "Nikmati momen kecil yang membuatmu tersenyum. 🌻",
    "😌 Tenang": "Kamu tidak harus terburu-buru menjalani semuanya. 🍃",
    "😔 Sedih": "Tidak apa-apa jika hari ini terasa berat. Beri dirimu waktu. 🤍",
    "😫 Lelah": "Beristirahat juga bagian dari perjalananmu. 🌙",
    "😰 Cemas": "Jalani satu langkah terlebih dahulu. Kamu tidak harus mengetahui semua jawaban hari ini. 🫶",
    "😡 Kesal": "Ambil jeda sejenak dan beri ruang untuk perasaanmu. 🌱"
}

history = []
selected_mood = {"value": None}
current_theme = "Soft Sage (Default)"
current_volume = 0.35  # Volume awal default

# =====================================
# 2. OUTPUT & STYLE DYNAMIC CONTAINER
# =====================================

style_output = widgets.Output()
header_output = widgets.Output()

def update_theme_styles(theme_name):
    t = themes[theme_name]
    with style_output:
        style_output.clear_output(wait=True)
        display(HTML(f"""
        <style>
        /* Global container style with smooth transition */
        .global-app-container {{
            background-color: {t['bg_main']} !important;
            border: 2px solid {t['border']} !important;
            color: {t['text_main']} !important;
            padding: 24px;
            border-radius: 22px;
            font-family: Arial, sans-serif;
            transition: background-color 0.8s ease, color 0.8s ease, border-color 0.8s ease, box-shadow 0.8s ease;
            box-shadow: 0 10px 25px rgba(0,0,0,0.02);
        }}
        .header-banner {{
            text-align: center;
            margin-bottom: 20px;
        }}
        .mood-choice button {{
            background-color: {t['button_bg']} !important;
            color: {t['button_text']} !important;
            border: 1px solid {t['border']} !important;
            font-weight: bold !important;
            border-radius: 10px !important;
            transition: background-color 0.5s ease, color 0.5s ease, border-color 0.5s ease;
        }}
        .mood-choice button:hover {{
            background-color: {t['button_hover']} !important;
        }}
        .widget-textarea textarea {{
            border: 1px solid {t['border']} !important;
            background-color: {t['bg_main']} !important;
            color: {t['text_main']} !important;
            border-radius: 12px;
            transition: background-color 0.8s ease, color 0.8s ease, border-color 0.8s ease;
        }}

        /* Styling dropdown widgets to look like bubbles with smooth transitions */
        .bubble-dropdown select {{
            background-color: {t['bubble_bg']} !important;
            color: {t['text_main']} !important;
            border: 1.5px solid {t['border']} !important;
            border-radius: 20px !important;
            padding: 4px 12px !important;
            font-weight: bold !important;
            cursor: pointer;
            box-shadow: 0 2px 5px rgba(0,0,0,0.05);
            outline: none;
            transition: background-color 0.8s ease, color 0.8s ease, border-color 0.8s ease, transform 0.2s;
        }}
        .bubble-dropdown select:hover {{
            border-color: {t['text_main']} !important;
            transform: scale(1.02);
        }}
        .bubble-dropdown label {{
            color: {t['text_main']} !important;
            font-weight: bold !important;
            transition: color 0.8s ease;
        }}
        </style>
        """))

    with header_output:
        header_output.clear_output(wait=True)
        display(HTML(f"""
        <div class="header-banner">
            <div style="font-size:36px;">🌿 ☁️ 🌷</div>
            <h2>How Are You Today?</h2>
            <p>Luangkan waktu sejenak untuk memahami dirimu.</p>
            <b>Bagaimana perasaanmu hari ini?</b>
        </div>
        """))

# =====================================
# 3. KOMPONEN INTERAKTIF
# =====================================

result = widgets.Output()
history_view = widgets.Output()
save_result = widgets.Output()

note = widgets.Textarea(
    placeholder="Tuliskan apa yang sedang kamu rasakan...",
    layout=widgets.Layout(width="100%", height="90px")
)
note.add_class("widget-textarea")

buttons = []

def choose_mood(button):
    selected_mood["value"] = button.description
    t = themes[current_theme]

    with result:
        result.clear_output(wait=True)
        display(HTML(f"""
        <div style="
            background:{t['bubble_bg']};
            padding:18px;
            border-radius:15px;
            color:{t['text_main']};
            border: 1px solid {t['border']};
            line-height:1.7;
            margin: 15px 0;
            transition: background-color 0.8s ease, color 0.8s ease, border-color 0.8s ease;">
            <h3>Perasaanmu: {button.description}</h3>
            <p>{moods[button.description]}</p>
            <p><i>Terima kasih sudah meluangkan waktu untuk dirimu. 💚</i></p>
        </div>
        """))

for mood in moods:
    button = widgets.Button(
        description=mood,
        layout=widgets.Layout(
            width="155px",
            height="44px",
            margin="4px"
        )
    )
    button.add_class("mood-choice")
    button.on_click(choose_mood)
    buttons.append(button)

row1 = widgets.Box(
    children=buttons,
    layout=widgets.Layout(
        display='flex',
        flex_flow='row wrap',
        justify_content='center'
    )
)

# =====================================
# 4. FUNGSI MENAMPILKAN RIWAYAT
# =====================================

def show_history():
    t = themes[current_theme]
    with history_view:
        history_view.clear_output(wait=True)

        if not history:
            print("Belum ada refleksi yang disimpan. 🌱")
            return

        for item in reversed(history):
            display(HTML(f"""
            <div style="
                background:{t['bubble_bg']};
                border:1px solid {t['border']};
                border-radius:12px;
                padding:14px;
                margin:8px 0;
                color:{t['text_main']};
                transition: background-color 0.8s ease, color 0.8s ease, border-color 0.8s ease;">
                <b>{item['waktu']}</b>
                <h4>{item['perasaan']}</h4>
                <p>{item['pesan']}</p>
                <p><b>Catatan:</b> {item['catatan_html']}</p>
            </div>
            """))

# =====================================
# 5. FUNGSI MENYIMPAN REFLEKSI
# =====================================

save_button = widgets.Button(
    description="Simpan refleksi",
    icon="heart",
    button_style="success",
    layout=widgets.Layout(width="180px", height="40px")
)

def save_reflection(button):
    with save_result:
        save_result.clear_output(wait=True)

        if selected_mood["value"] is None:
            print("Pilih suasana hati terlebih dahulu, ya. 🌷")
            return

        mood = selected_mood["value"]
        catatan = note.value.strip()

        import html
        safe_note = html.escape(catatan) if catatan else "(Belum ada catatan)"

        entry = {
            "waktu": datetime.now().strftime("%d-%m-%Y %H:%M"),
            "perasaan": mood,
            "pesan": moods[mood],
            "catatan": catatan,
            "catatan_html": safe_note
        }

        history.append(entry)

        print("Refleksi berhasil disimpan ke riwayat! 💚")
        print("Perasaan:", mood)
        print("Catatan:", catatan or "(Belum ada catatan)")

        show_history()

save_button.on_click(save_reflection)

# =====================================
# 6. TOMBOL UNTUK MELIHAT RIWAYAT
# =====================================

history_button = widgets.Button(
    description="Lihat riwayat refleksi",
    icon="book",
    layout=widgets.Layout(width="220px", height="40px")
)

def view_history(button):
    show_history()

history_button.on_click(view_history)

row2 = widgets.VBox([
    widgets.HTML("<b>Tulis catatan tambahan:</b>"),
    note,
    widgets.HBox([save_button, history_button], layout=widgets.Layout(justify_content='center', margin='10px 0px'))
])

# =====================================
# 6.2 WIDGET GANTI TEMA
# =====================================

theme_dropdown = widgets.Dropdown(
    options=list(themes.keys()),
    value=current_theme,
    description='Tema Warna:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px', margin='5px')
)
theme_dropdown.add_class("bubble-dropdown")

def on_theme_change(change):
    global current_theme
    current_theme = change['new']
    update_theme_styles(current_theme)
    if selected_mood["value"]:
        for b in buttons:
            if b.description == selected_mood["value"]:
                choose_mood(b)
    if history:
        show_history()

theme_dropdown.observe(on_theme_change, names='value')

update_theme_styles(current_theme)

# =====================================
# 6.5 AUDIO ENGINE (NATIVE TRAVERSAL CONTROL)
# =====================================

songs = {
    "Calm Melody": [
        (523.25, 1.2), (392.00, 1.2), (440.00, 1.2), (329.63, 1.2),
        (349.23, 1.2), (392.00, 1.2), (293.66, 1.2), (261.63, 2.0)
    ],
    "Joyful Breeze": [
        (329.63, 0.8), (392.00, 0.8), (523.25, 0.8), (587.33, 0.8),
        (659.25, 1.2), (587.33, 0.8), (523.25, 1.6)
    ],
    "Deep Meditation": [
        (220.00, 2.5), (196.00, 2.5), (220.00, 2.5), (261.63, 3.0),
        (174.61, 2.5), (196.00, 2.5), (146.83, 4.0)
    ]
}

def create_audio_data(song_name):
    sample_rate = 22050
    melody = songs[song_name]
    audio_parts = []
    for frequency, duration in melody:
        t = np.arange(int(sample_rate * duration)) / sample_rate
        tone = np.sin(2 * np.pi * frequency * t) + 0.18 * np.sin(2 * np.pi * frequency * 2 * t)
        fade_length = int(sample_rate * 0.2)
        envelope = np.ones(len(t))
        envelope[:fade_length] = np.linspace(0, 1, fade_length)
        envelope[-fade_length:] = np.linspace(1, 0, fade_length)
        audio_parts.append(tone * envelope)
    audio = np.concatenate(audio_parts)
    audio = audio / (np.max(np.abs(audio)) + 1e-9)
    audio = audio * 0.35
    audio_int16 = (audio * 32767).astype(np.int16)
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(audio_int16.tobytes())
    return buffer.getvalue()

music_output = widgets.Output()

music_dropdown = widgets.Dropdown(
    options=list(songs.keys()),
    value="Calm Melody",
    description='Musik Latar:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='190px', margin='5px')
)
music_dropdown.add_class("bubble-dropdown")

def play_song(song_name):
    music_data = create_audio_data(song_name)
    music_base64 = base64.b64encode(music_data).decode("utf-8")
    t = themes[current_theme]
    with music_output:
        music_output.clear_output(wait=True)
        display(HTML(f"""
        <div class="music-control-wrapper" style="display: inline-flex; align-items: center; margin: 5px; padding: 6px 14px; background: {t['bubble_bg']}; border-radius: 20px; border: 1.5px solid {t['border']}; font-family: sans-serif; box-shadow: 0 2px 5px rgba(0,0,0,0.05); transition: background-color 0.8s ease, border-color 0.8s ease;">
            <span style="margin-right: 8px; font-weight: bold; color: {t['text_main']}; transition: color 0.8s ease;">🔊 Vol:</span>
            <input type="range" min="0" max="1" step="0.02" value="{current_volume}"
                   style="width: 100px; cursor: pointer; accent-color: {t['text_main']}; transition: accent-color 0.8s ease;"
                   oninput="this.parentNode.querySelector('audio').volume = this.value;" />
            <audio autoplay loop style="display:none;">
                <source src="data:audio/wav;base64,{music_base64}" type="audio/wav">
            </audio>
        </div>
        <script>
            (function() {{
                var wrapper = document.querySelector('.music-control-wrapper');
                if (wrapper) {{
                    var audioEl = wrapper.querySelector('audio');
                    var sliderEl = wrapper.querySelector('input[type=range]');
                    if (audioEl && sliderEl) {{
                        audioEl.volume = parseFloat(sliderEl.value);
                    }}
                }}
            }})();
        </script>
        """))

def on_music_change(change):
    global current_volume
    with music_output:
        display(HTML("""
        <script>
            var slider = document.querySelector('.music-control-wrapper input[type=range]');
            if (slider) {
                google.colab.kernel.invokeFunction('update_volume_var', [parseFloat(slider.value)], {});
            }
        </script>
        """))
    play_song(change['new'])

def update_volume_var(vol_val):
    global current_volume
    current_volume = vol_val

from google.colab import output
output.register_callback('update_volume_var', update_volume_var)

music_dropdown.observe(on_music_change, names='value')

play_song("Calm Melody")

# =====================================
# 7. MENAMPILKAN APLIKASI
# =====================================

main_container = widgets.VBox([
    header_output,
    row1,
    result,
    widgets.HTML("""
        <h4 style='color: inherit; margin-top: 15px;'>
            Catatan kecil untuk hari ini 🌷
        </h4>
    """),
    row2,
    save_result,
    history_view
])

main_container.add_class("global-app-container")

display(widgets.VBox([
    style_output,
    widgets.HBox([theme_dropdown, music_dropdown, music_output], layout=widgets.Layout(justify_content='center', margin='10px 0', align_items='center')),
    main_container
]))